# Constructores alternativos con `@classmethod`

El uso **estrella** de los `@classmethod` son los **constructores alternativos**,
es un patrón que aparece constantemente en el Python real.

La idea: muchas veces un mismo objeto se puede crear a partir de **fuentes de datos
distintas**. Una fecha, por ejemplo, podríamos construirla desde día/mes/año, desde
un texto como `"2026-06-19"`, desde un *timestamp*, o simplemente pidiendo "la fecha
de hoy".

En este vídeo veremos:

- Por qué meter "mil formas de construir" en el `__init__` es mala idea.
- Cómo usar `@classmethod` para crear **constructores alternativos** limpios.
- Cómo aprovecharlos para **validar** los datos de entrada.
- Dónde aparece este patrón en librerías que ya usas.

---

## 1. El problema: un `__init__` que intenta con todo

Imagina que queremos poder crear una `Fecha` de varias formas. Una tentación es
llenar el constructor de parámetros opcionales y condicionales:

In [1]:
class Fecha:
    def __init__(self, dia=None, mes=None, anio=None, texto=None, timestamp=None):
        if texto is not None:
            anio, mes, dia = map(int, texto.split("-"))
        elif timestamp is not None:
            from datetime import datetime

            d = datetime.fromtimestamp(timestamp)
            dia, mes, anio = d.day, d.month, d.year

        self.dia = dia
        self.mes = mes
        self.anio = anio


# Funciona, pero la llamada es confusa: ¿qué significan estos argumentos?
f = Fecha(texto="2026-06-19")
print(f.dia, f.mes, f.anio)

19 6 2026


Esto funciona, pero es un **mal diseño**:

- El `__init__` mezcla varias responsabilidades y se llena de `if/elif`.
- Quien llama tiene que recordar combinaciones raras de parámetros.
- Es fácil pasar argumentos incompatibles (`Fecha(dia=5, texto="...")`) sin que nadie
  avise.

Hay una forma mucho más limpia.

---

## 2. La solución: un `__init__` simple + constructores alternativos

La idea del patrón es:

- El `__init__` se queda **simple y canónico**: recibe los datos "de verdad" que
  necesita el objeto (aquí, día, mes y año).
- Cada forma alternativa de construir se convierte en un **`@classmethod`** con un
  nombre descriptivo, que prepara los datos y llama al constructor.

In [3]:
from datetime import date, datetime


class Fecha:
    def __init__(self, dia, mes, anio):  # constructor canónico: simple y claro
        self.dia = dia
        self.mes = mes
        self.anio = anio

    def print_fecha(self):
        return f"Fecha({self.dia:02d}/{self.mes:02d}/{self.anio})"

    @classmethod
    def desde_string(cls, texto):  # "2026-06-19"
        anio, mes, dia = map(int, texto.split("-"))
        return cls(dia, mes, anio)

    @classmethod
    def desde_timestamp(cls, ts):
        d = datetime.fromtimestamp(ts)
        return cls(d.day, d.month, d.year)

    @classmethod
    def hoy(cls):
        d = date.today()
        return cls(d.day, d.month, d.year)

In [4]:
# Ahora cada forma de crear una fecha tiene un nombre que se lee solo
fecha = Fecha(19, 6, 2026)
fecha.print_fecha()

fecha = Fecha.desde_string("2026-06-19")
fecha.print_fecha()

fecha = Fecha.desde_timestamp(1_700_000_000)
fecha.print_fecha()

fecha = Fecha.hoy()
fecha.print_fecha()

'Fecha(27/06/2026)'

Compara la legibilidad: `Fecha.desde_string("2026-06-19")` dice **exactamente** lo que
hace. No hay que adivinar qué parámetros del `__init__` rellenar.

> 💡 Por eso a estos métodos se les llama también ***factory methods*** (métodos
> fábrica): son "pequeñas fábricas" que saben construir el objeto a partir de una
> fuente concreta y siempre devuelven una instancia ya lista para usar.

> 💡 Fíjate en que dentro de cada `@classmethod` devolvemos **`cls(...)`** y no
> `Fecha(...)`. `cls` es "la clase sobre la que se llamó el método": usarlo hace que
> el patrón siga funcionando bien aunque alguien herede de `Fecha` (lo veremos en
> detalle más adelante).

---

## 3. Aprovechar para validar

Como cada constructor alternativo es su propio método, es el sitio natural para
**validar** la fuente de datos antes de crear el objeto. Por ejemplo, avisar de un
string mal formado con un mensaje claro:

In [5]:
class Fecha:
    def __init__(self, dia, mes, anio):
        self.dia = dia
        self.mes = mes
        self.anio = anio

    def print_fecha(self):
        return f"Fecha({self.dia:02d}/{self.mes:02d}/{self.anio})"

    @classmethod
    def desde_string(cls, texto):
        partes = texto.split("-")
        if len(partes) != 3:
            raise ValueError(f"Formato esperado 'AAAA-MM-DD', recibido: {texto!r}")
        anio, mes, dia = map(int, partes)
        return cls(dia, mes, anio)


fecha = Fecha.desde_string("2026-06-19")
fecha.print_fecha()

try:
    fecha = Fecha.desde_string("19/06/2026")  # formato incorrecto
    fecha.print_fecha()
except ValueError as e:
    print("Error:", e)

Error: Formato esperado 'AAAA-MM-DD', recibido: '19/06/2026'


---

## 4. No es un invento: ya lo usas

Este patrón está por toda la librería estándar y en librerías muy populares. Cuando
veas un método de la forma `Tipo.desde_algo(...)` o `Tipo.from_algo(...)`, casi
siempre es un constructor alternativo:

| Llamada | Qué construye |
|---------|----------------|
| `dict.fromkeys(["a", "b"], 0)` | Un diccionario a partir de unas claves |
| `datetime.fromisoformat("2026-06-19")` | Un `datetime` a partir de un texto ISO |
| `datetime.fromtimestamp(ts)` | Un `datetime` a partir de un *timestamp* |
| `int.from_bytes(b"\\x01", "big")` | Un entero a partir de unos bytes |

Reconocer el patrón te ayuda a **leer** mejor código ajeno y a **diseñar** APIs más
claras en el tuyo.

In [ ]:
# Ejemplos reales de constructores alternativos de la librería estándar
from datetime import datetime

print(dict.fromkeys(["a", "b", "c"], 0))
print(datetime.fromisoformat("2026-06-19"))

---

## 5. Resumen

En este vídeo hemos aprendido un patrón que aparece por todas partes:

1. Cuando un objeto se puede crear desde **varias fuentes**, no infles el `__init__`
   con parámetros opcionales y condicionales.
2. Deja un `__init__` **simple y canónico**, y añade un **`@classmethod`** por cada
   forma alternativa de construir, con un **nombre descriptivo**
   (`desde_string`, `hoy`, ...).
3. Dentro del `@classmethod`, devuelve **`cls(...)`** en lugar de escribir el nombre
   de la clase a mano.
4. Cada constructor alternativo es el sitio ideal para **validar** su fuente de datos.
5. Es un patrón estándar (*factory method*): lo ves en `dict.fromkeys`,
   `datetime.fromisoformat`, etc.

| Concepto | Idea |
|----------|------|
| Constructor canónico | `__init__` simple con los datos esenciales |
| Constructor alternativo | `@classmethod` que prepara los datos y llama a `cls(...)` |
| Nombre descriptivo | `desde_string`, `desde_timestamp`, `hoy`... |